# Full Extraction Pipeline

Run **Kernel → Restart & Run All** for a complete automated run.

Edit **Cell 1** only: sample size, target IDs, model.

## Cell 1 — Configuration (edit this)

In [27]:
import os, json, time, re
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from collections import Counter
import anthropic

# ── Load credentials and paths ─────────────────────────────────────────────
with open("Claude_API.txt") as f:
    api_key = f.read().strip()
os.environ['ANTHROPIC_API_KEY'] = api_key

config = {}
with open("File_Directory.txt") as f:
    for line in f:
        if ":" in line:
            key, val = line.split(":", 1)
            config[key.strip()] = val.strip()

INPUT_FILE = Path(config["Input"])
OUTPUT_DIR = Path(config["Output"])

# ── EDIT THESE ─────────────────────────────────────────────────────────────
MODEL       = "claude-sonnet-4-6"
SAMPLE_SIZE = 50       # set to None to run all funds
RANDOM_STATE = 42
TARGET_IDS  =  None

OBJECTIVE_COLUMNS = [
    'PRIIPS KID Objective',
    'KIID Objective/Investment Policy',
    'Prospectus Objective',
    'Investment Strategy - English',
    'PRIIPS KID Objective - Danish',
    'PRIIPS KID Objective - Dutch',
    'PRIIPS KID Objective - Finnish',
    'PRIIPS KID Objective - French',
    'PRIIPS KID Objective - German',
    'PRIIPS KID Objective - Italian',
    'PRIIPS KID Objective - Norwegian',
    'PRIIPS KID Objective - Portuguese',
    'PRIIPS KID Objective - Spanish',
    'PRIIPS KID Objective - Swedish',
    'KIID Objective/Investment Policy - German',
    'KIID Objective/Investment Policy - French',
    'KIID Objective/Investment Policy - Italian',
    'KIID Objective/Investment Policy - Spanish',
    'KIID Objective/Investment Policy - Norwegian',
    'KIID Objective/Investment Policy - Swedish',
    'KIID Objective/Investment Policy - Finnish',
    'KIID Objective/Investment Policy - Portuguese',
    'KIID Objective/Investment Policy - Danish',
    'Investment Strategy - Danish',
    'Investment Strategy - Finnish',
    'Investment Strategy - French',
    'Investment Strategy - German',
    'Investment Strategy - Italian',
    'Investment Strategy - Norwegian',
    'Investment Strategy - Portuguese',
    'Investment Strategy - Spanish',
    'Investment Strategy - Swedish',
    'Strategy Description',
]

print("Configuration loaded.")
print(f"  Input:  {INPUT_FILE}")
print(f"  Output: {OUTPUT_DIR}")
print(f"  Model:  {MODEL}")


Configuration loaded.
  Input:  /Users/dannyhogan/Desktop/Hogan_RA_Work/Download Sustainable Funds 2026-04-15.xlsx
  Output: /Users/dannyhogan/Desktop/Hogan_RA_Work
  Model:  claude-sonnet-4-6


## Cells 2–4 — Prompts (update when prompts change)

In [28]:
PASS1_SYSTEM_PROMPT = """You are extracting fund objectives from regulatory disclosure text for European mutual funds.


CORE EXTRACTION PRINCIPLE:

Extract WHAT the fund aims to achieve for investors.
Strip everything that describes HOW.

HOW covers:
- Implementation: what the fund invests in, asset allocation,
  geographic or sector focus, types of securities
- Mechanism: how the goal is pursued ("by investing in...",
  "through active management...", "through exposure to...",
  "by selecting...", "through a strategy of...",
  "on a [diversified] portfolio of [asset class]...")
- Measurement: how performance, ESG scores, or carbon intensity
  are calculated or tracked
- Regulation: SFDR compliance language, minimum allocation
  requirements, mandatory portfolio composition rules
- Constraints: self-imposed rules the portfolio must satisfy —
  ESG score thresholds, carbon intensity limits, minimum shares
  of sustainable investments
- Exclusions: types of firms, activities, or industries the fund
  avoids

The HOW list above is illustrative, not exhaustive. The underlying
principle: a clause is HOW if removing it would not change what
the fund is trying to achieve for investors, only the manner in
which or where it pursues it.

You will receive MULTIPLE columns of text for a single fund. Extract objectives from EACH column INDEPENDENTLY.
Do NOT cross-reference between columns. Treat each column as a standalone source.

WHAT IS A FUND OBJECTIVE:
A fund objective is what the fund is trying to achieve THROUGH its investments — the intended outcome for investors.

This means: financial return goals (capital growth, income, outperforming a benchmark) and sustainability outcomes the fund has set as explicit goals.

This does NOT include, on the basis of content alone:
- What the fund invests in: strategy, asset classes, sectors, types of companies
- Commitments to invest in or avoid certain types of firms or industries
- Portfolio constraints the fund imposes on itself — e.g. maintaining an ESG score above a benchmark, keeping carbon intensity below a threshold, requiring minimum shares of sustainable investments
- Exclusions of firm types, activities, or industries

EXPLICIT LABELING OVERRIDE:
If the fund document uses indicator language to explicitly label something as an objective, extract it regardless of whether it would otherwise be excluded as a constraint or investment policy.

Indicator language: "the objective is", "the fund aims to", "the fund seeks to", "the fund's objective includes", "the fund targets", "the fund's investment objective is"

Example: "The fund objective is capital growth and lower GHG emissions than the benchmark."
→ Extract BOTH. "Lower GHG emissions than the benchmark" is a portfolio constraint but the document labels it as an objective, so extract it.

This rule avoids biasing extraction against sustainability objectives. Distinguishing genuine objectives from portfolio constraints is handled in a downstream classification step.

MULTIPLE OBJECTIVES:
There may be more than one objective per column. Extract ALL and label them separately.
Split objectives when joined by "and", "and/or", "while", "which also", "that also", or similar.
For "and/or" constructions, treat each alternative as a distinct objective.
If one part states a financial goal and another states a sustainability goal, split them.
Examples:
- "provide income and moderate capital growth" → two objectives: "provide income" and "achieve moderate capital growth"
- "exceed the performance of the index while maintaining a higher ESG score" → two objectives
- "achieve capital growth and/or continuous returns" → two objectives: "achieve capital growth" and "achieve continuous returns"
- "long-term capital growth and the generation of a market-appropriate return" → two objectives: "long-term capital growth" and "achieve a market-appropriate return"
- "targeting a significantly reduced risk of loss and significantly lower volatility compared to the general equity market" → two objectives: "reduce risk of capital loss" and "maintain lower volatility than the general equity market"

When stripping HOW and splitting in the same sentence, process each clause independently before finalising:
"Increase the value of your investment (mainly through capital growth) and outperform the benchmark over the recommended holding period"
→ Clause 1: "Increase the value of your investment" — strip HOW "(mainly through capital growth)"
→ Clause 2: "outperform the benchmark" — drop generic time horizon "over the recommended holding period"
→ Objective 1: "increase the value of your investment"
→ Objective 2: "outperform the benchmark"

HOW clauses do not appear as coordinate "and" clauses parallel to the main objective. When a phrase follows "and" as the second part of a compound objective statement, treat it as a second WHAT candidate — evaluate it independently before applying any stripping.

Nominalized goal phrases — "the generation of X", "the achievement of X", "the maximisation of X", "the preservation of X", "the realisation of X" — that follow "and" are noun-form objective statements. Rephrase them as clean verb-form objectives: "generation of a market-appropriate return" → "achieve a market-appropriate return".

"while" clauses — critical distinction:
- "while REDUCING / MINIMISING risk" = a second objective → split and extract separately
- "while ACCEPTING / TOLERATING [higher] risk" = a risk tolerance descriptor → exclude entirely

"Income and capital growth" is always two distinct objectives — income refers to cash distributions or yield; capital growth refers to price appreciation. Always split.
- "achieve income and capital growth over the medium to long term" → two objectives: "achieve income" and "achieve capital growth over the medium to long term"
Exception: When income and capital growth appear inside a HOW phrase ("through a combination of capital growth and income"), strip the entire HOW phrase — do not split.
- "achieve a total return through a combination of capital growth and income" → one objective: "achieve a total return"

For what qualifies as "sustainable" vs "sustainable_disclosure", see SUSTAINABILITY CONTENT below.

DO NOT INCLUDE:
- Investment policy/strategy: what the fund invests in, how securities are selected, asset allocation
- Mechanism or investment vehicle: "by investing in...", "through active management...", "on a portfolio of..."
- Company-activity descriptions: "invest in companies that...", "companies whose products..." — even if labeled "sustainable investment objective", extract only what the fund itself aims to achieve
- Investment philosophy statements or general descriptions of the fund's investment approach
- "while taking into account ESG criteria" or "taking into account the risk level" = not an objective
- Risk information, distribution/dividend policy
- Benchmark references used solely for comparison (not as a target to beat)
- Duplicate objectives within the same column

SUSTAINABILITY CONTENT — WHAT TO EXTRACT VS WHAT TO EXCLUDE:

This is the most important judgment call in the extraction. Apply these rules in order:

The key test: Does the text describe something the fund COMMITS TO ACHIEVING (an outcome, a target, a minimum allocation) or something the fund TAKES INTO ACCOUNT (a process, a consideration, a methodology)? Extract the former, exclude the latter — but also apply the sustainable vs sustainable_disclosure distinction in Step 2 below.

Step 1 — Is it pure SFDR Article 8/9 boilerplate?
These phrases are required regulatory language and are NEVER extracted — not as "sustainable", not as "sustainable_disclosure", not at all:
- "promotes environmental and/or social characteristics"
- "is promoting ESG characteristics"
- "is classified as Article 8/9 under SFDR"
If the text contains ONLY these generic phrases with no additional specific content, there is no extraction. Move to Step 2 only if the text goes beyond these phrases with a specific commitment, target, or allocation figure.

Step 2 — Does the text go beyond boilerplate with a specific sustainable commitment?
If yes, determine whether it is a genuine independently-set objective or a regulatory disclosure statement — and classify accordingly as "sustainable" or "sustainable_disclosure".

Use "sustainable" when the fund has explicitly labeled a sustainability goal as an objective using indicator language, OR when the text describes a genuine sustainability outcome (not a portfolio constraint):
- Named environmental/social outcomes the fund commits to achieving: GHG reductions, biodiversity improvements, SDG contributions, positive social impact
- Specific solidarity commitments with a named % allocation
- Criteria for good governance and sustainable development framed as a fund-level goal

EXTRACT as "sustainable":
- "contribute to reducing greenhouse gas emissions" → sustainable (outcome)
- "positive impact on environment and social objectives" → sustainable (outcome)
- "integrating criteria for good governance and sustainable development" → sustainable (outcome)
- "solidarity investments of 5-10% in approved solidarity enterprises" → sustainable (commitment)
- "The fund aims to maintain a carbon footprint 5% lower than the Index" → sustainable (indicator language: "aims to")
- "The fund's objective includes a higher ESG score than the benchmark" → sustainable (indicator language: "objective includes")

Do NOT extract without indicator language — these are portfolio constraints, not objectives:
- "the fund maintains a carbon footprint 5% lower than the Index" → do not extract (no indicator language)
- "the fund's ESG score is higher than the benchmark" → do not extract (no indicator language)
- "the fund keeps carbon intensity below the index" → do not extract (no indicator language)

Use "sustainable_disclosure" when the text uses regulatory language that may be a mandatory disclosure rather than a fund-chosen objective — specifically when it:
- References any regulation, law, or article by name or number ("as defined under SFDR", "in accordance with Article 8/9", "pursuant to Regulation (EU) 2019/2088", "pursuant to Article L.3332-17-1 of the Labour Code", "under Article X of [any law]")
- States only a generic minimum allocation without a specific fund-set % figure (e.g. "a minimum share" with no number)
- Reads as a portfolio composition rule rather than a stated goal

EXTRACT as "sustainable_disclosure":
- "invests at least X% of assets in Sustainable Investments, as defined under SFDR" → sustainable_disclosure (regulatory reference language)
- "maintains a minimum share of sustainable investments" [no specific % stated] → sustainable_disclosure (generic minimum, no fund-set target)
- "Between 5% and 10% of assets are invested in approved solidarity enterprises pursuant to Article L.3332-17-1 of the Labour Code" → sustainable_disclosure (legal article citation)

Do not extract process descriptions: "taking into account ESG criteria", "considering sustainability risks", "ESG integration", "employs ESG criteria in stock selection" — these describe methodology, not outcomes. Generic "promotes environmental and/or social characteristics" is covered by Step 1 above.

Step 3 — Does the fund explicitly disclaim sustainable objectives?
If the text states "the fund does not have sustainable investment as its objective" and the sustainability content is framed purely as an approach or consideration, do not extract a sustainable objective.

Worked examples:
- "The fund promotes environmental and social characteristics and maintains a minimum share of sustainable investments in accordance with Article 8 of the EU SFDR." → EXTRACT as SUSTAINABLE_DISCLOSURE; objective_text: "maintain a minimum share of sustainable investments"
- "The objective is to achieve outperformance while integrating criteria for good governance and sustainable development." → TWO objectives: (1) financial: "outperform the benchmark", (2) sustainable: "integrate criteria for good governance and sustainable development"
- "The fund invests 5-10% of its assets in approved solidarity enterprises." → EXTRACT as SUSTAINABLE; objective_text: "invest 5-10% of assets in solidarity enterprises"
- "The Sub-Fund invests at least 50% of assets in Sustainable Investments, as defined under SFDR." → EXTRACT as SUSTAINABLE_DISCLOSURE
- "The fund targets a carbon footprint at least 5% lower than the Index." → EXTRACT as SUSTAINABLE — "targets" is indicator language labeling this as an objective.
- "The fund maintains a carbon footprint at least 5% lower than the Index." → Do NOT extract — describes a portfolio constraint without indicator language labeling it as an objective.
- "The fund invests in companies whose products contribute to the SDGs." → Do NOT extract — company description, not fund objective.

TIME HORIZON:
Include SPECIFIC time horizons that represent a fund's stated investment horizon: "long-term", "medium-term", "over 5 years", "over a rolling 3-year period".
Exclude GENERIC regulatory language that conveys no fund-specific information: "over the recommended investment period", "over the recommended holding period", "over a multi-year period". These are standard disclosure phrasing, not fund-chosen commitments.

EXTRACTION RULES:
1. Write a concise, clean English statement of the objective — paraphrase if needed to remove scaffolding language and HOW content. Do not copy the full source sentence.
2. Scaffolding to strip from objective_text: "The fund aims to", "The investment objective is to", "The objective of the fund is to", "The Sub-Fund seeks to" — start directly with the goal verb or noun.
3. Record the verbatim source text in source_text — this is 1-3 sentences copied exactly from the source column, in the original language, showing where the objective was identified.
4. Classify each objective as "financial", "sustainable", or "sustainable_disclosure".
5. If no objective can be identified in a column, return an empty list for that column.
6. Detect the language of each column and record it.

OUTPUT FORMAT:
Return a JSON object where each key is the exact column name, and the value is:
{
  "language": "English" or "French" or "German" etc.,
  "objectives": [
    {
      "objective_text": "concise English statement of the goal — no full sentences, no scaffolding, no HOW",
      "source_text": "verbatim 1-3 sentence excerpt from source in original language",
      "objective_type": "financial" or "sustainable" or "sustainable_disclosure"
    }
  ]
}

If a column has no identifiable objective:
{
  "language": "English",
  "objectives": []
}

IMPORTANT: When extracting verbatim source_text that contains quotation marks (including German \u201e...\u201c quotes, French \u00ab...\u00bb quotes, or any other quotation marks), replace them with single quotes. This is critical to ensure valid JSON output.

"""

PASS1_FEW_SHOT = [
    {
        "fund_name": "Example Multi-Column Fund",
        "columns": {
            "PRIIPS KID Objective": "The Fund aims to maximise the return on your investment through a combination of capital growth and income on the Fund's assets and invest in a manner consistent with the principles of environmental, social and governance (ESG) investing. The Fund invests globally at least 70% of its total assets in the equity securities of companies the main business of which is financial services.",
            "PRIIPS KID Objective - French": "Le Fonds vise \u00e0 maximiser le rendement de votre investissement par une combinaison de croissance du capital et de revenus sur les actifs du Fonds et \u00e0 investir d'une mani\u00e8re conforme aux principes de l'investissement environnemental, social et de gouvernance (ESG). Le Fonds investit \u00e0 l'\u00e9chelle mondiale au moins 70 % de son actif total dans les titres de participation de soci\u00e9t\u00e9s dont l'activit\u00e9 principale est les services financiers."
        },
        "response": {
            "PRIIPS KID Objective": {
                "language": "English",
                "objectives": [
                    {
                        "objective_text": "maximise the return on your investment",
                        "source_text": "The Fund aims to maximise the return on your investment through a combination of capital growth and income on the Fund's assets and invest in a manner consistent with the principles of environmental, social and governance (ESG) investing.",
                        "objective_type": "financial"
                    }
                ]
            },
            "PRIIPS KID Objective - French": {
                "language": "French",
                "objectives": [
                    {
                        "objective_text": "maximise the return on your investment",
                        "source_text": "Le Fonds vise \u00e0 maximiser le rendement de votre investissement par une combinaison de croissance du capital et de revenus sur les actifs du Fonds.",
                        "objective_type": "financial"
                    }
                ]
            }
        }
    },
    {
        "fund_name": "Example Sustainability Split Fund",
        "columns": {
            "PRIIPS KID Objective": "The fund seeks to achieve capital growth and to outperform the benchmark. The fund's sustainable investment objective is to contribute to reducing greenhouse gas emissions. The fund also aims to have long-term positive impact on environment and social objectives."
        },
        "response": {
            "PRIIPS KID Objective": {
                "language": "English",
                "objectives": [
                    {
                        "objective_text": "achieve capital growth",
                        "source_text": "The fund seeks to achieve capital growth and to outperform the benchmark.",
                        "objective_type": "financial"
                    },
                    {
                        "objective_text": "outperform the benchmark",
                        "source_text": "The fund seeks to achieve capital growth and to outperform the benchmark.",
                        "objective_type": "financial"
                    },
                    {
                        "objective_text": "contribute to reducing greenhouse gas emissions",
                        "source_text": "The fund's sustainable investment objective is to contribute to reducing greenhouse gas emissions.",
                        "objective_type": "sustainable"
                    },
                    {
                        "objective_text": "have long-term positive impact on environment and social objectives",
                        "source_text": "The fund also aims to have long-term positive impact on environment and social objectives.",
                        "objective_type": "sustainable"
                    }
                ]
            }
        }
    },
    {
        "fund_name": "Example Norwegian Fund",
        "columns": {
            "PRIIPS KID Objective": "M\u00e5lsetting\n\nFondets m\u00e5lsetting er \u00e5 skape h\u00f8yest mulig relativ avkastning mot referanseindeksen, MSCI World AC, Net Total Return (m\u00e5lt i NOK).\n\nFondet skal investere i selskaper globalt som har l\u00f8sninger p\u00e5 FN's b\u00e6rekraftsm\u00e5l og dermed bidrar til omstillingen til et mer b\u00e6rekraftig samfunn."
        },
        "response": {
            "PRIIPS KID Objective": {
                "language": "Norwegian",
                "objectives": [
                    {
                        "objective_text": "create the highest possible relative return against the benchmark index, MSCI World AC, Net Total Return (measured in NOK)",
                        "source_text": "Fondets m\u00e5lsetting er \u00e5 skape h\u00f8yest mulig relativ avkastning mot referanseindeksen, MSCI World AC, Net Total Return (m\u00e5lt i NOK).",
                        "objective_type": "financial"
                    }
                ]
            }
        }
    },
    {
        "fund_name": "Example No-Objective Fund",
        "columns": {
            "PRIIPS KID Objective": "Management objective: Management takes as reference the profitability of the EUROSTOXX 50 Index, solely for informational or comparative purposes. Investment policy: Will invest more than 75% of total exposure in equity assets of European issuers."
        },
        "response": {
            "PRIIPS KID Objective": {
                "language": "English",
                "objectives": []
            }
        }
    },
    {
        "fund_name": "Example Company-Activity Exclusion Fund",
        "columns": {
            "KIID Objective/Investment Policy": "The fund aims to provide capital growth over the long term (5 years or more) by investing in US companies whose products and services are considered by the investment manager as contributing to positive environmental or social change and thereby have an impact on the development of a sustainable global economy."
        },
        "response": {
            "KIID Objective/Investment Policy": {
                "language": "English",
                "objectives": [
                    {
                        "objective_text": "provide capital growth over the long term (5 years or more)",
                        "source_text": "The fund aims to provide capital growth over the long term (5 years or more) by investing in US companies whose products and services are considered by the investment manager as contributing to positive environmental or social change.",
                        "objective_type": "financial"
                    }
                ]
            }
        }
    },
    {
        "fund_name": "Example And-Or Split Fund",
        "columns": {
            "PRIIPS KID Objective": "The Fund aims to achieve capital growth and/or continuous returns by investing in a diversified portfolio of global equities."
        },
        "response": {
            "PRIIPS KID Objective": {
                "language": "English",
                "objectives": [
                    {
                        "objective_text": "achieve capital growth",
                        "source_text": "The Fund aims to achieve capital growth and/or continuous returns by investing in a diversified portfolio of global equities.",
                        "objective_type": "financial"
                    },
                    {
                        "objective_text": "achieve continuous returns",
                        "source_text": "The Fund aims to achieve capital growth and/or continuous returns by investing in a diversified portfolio of global equities.",
                        "objective_type": "financial"
                    }
                ]
            }
        }
    },
    {
        "fund_name": "Example Nominalized Second Objective",
        "columns": {
            "PRIIPS KID Objective": "The investment objective of the fund is long-term capital growth and the generation of a market-appropriate return."
        },
        "response": {
            "PRIIPS KID Objective": {
                "language": "English",
                "objectives": [
                    {
                        "objective_text": "long-term capital growth",
                        "source_text": "The investment objective of the fund is long-term capital growth and the generation of a market-appropriate return.",
                        "objective_type": "financial"
                    },
                    {
                        "objective_text": "achieve a market-appropriate return",
                        "source_text": "The investment objective of the fund is long-term capital growth and the generation of a market-appropriate return.",
                        "objective_type": "financial"
                    }
                ]
            }
        }
    },
    {
        "fund_name": "Example Sector HOW Stripped",
        "columns": {
            "PRIIPS KID Objective": "The investment objective pursued by this fund consists of achieving reasonable capital growth through active direct and indirect investment in equities of the mining sector worldwide."
        },
        "response": {
            "PRIIPS KID Objective": {
                "language": "English",
                "objectives": [
                    {
                        "objective_text": "achieve reasonable capital growth",
                        "source_text": "The investment objective pursued by this fund consists of achieving reasonable capital growth through active direct and indirect investment in equities of the mining sector worldwide.",
                        "objective_type": "financial"
                    }
                ]
            }
        }
    },
    {
        "fund_name": "Example Portfolio Type HOW Stripped",
        "columns": {
            "PRIIPS KID Objective": "The objective of the Sub-Fund is to achieve attractive risk-adjusted returns on a diversified portfolio of private equity investments."
        },
        "response": {
            "PRIIPS KID Objective": {
                "language": "English",
                "objectives": [
                    {
                        "objective_text": "achieve attractive risk-adjusted returns",
                        "source_text": "The objective of the Sub-Fund is to achieve attractive risk-adjusted returns on a diversified portfolio of private equity investments.",
                        "objective_type": "financial"
                    }
                ]
            }
        }
    },
    {
        "fund_name": "Example Income And Capital Growth Split",
        "columns": {
            "PRIIPS KID Objective": "The Fund aims to achieve income and capital growth over the medium to long term."
        },
        "response": {
            "PRIIPS KID Objective": {
                "language": "English",
                "objectives": [
                    {
                        "objective_text": "achieve income over the medium to long term",
                        "source_text": "The Fund aims to achieve income and capital growth over the medium to long term.",
                        "objective_type": "financial"
                    },
                    {
                        "objective_text": "achieve capital growth over the medium to long term",
                        "source_text": "The Fund aims to achieve income and capital growth over the medium to long term.",
                        "objective_type": "financial"
                    }
                ]
            }
        }
    },
    {
        "fund_name": "Example Sustainable Disclosure Fund",
        "columns": {
            "PRIIPS KID Objective": "The fund seeks to achieve long-term capital growth. The Sub-Fund invests at least 50% of assets in Sustainable Investments, as defined under SFDR.",
            "PRIIPS KID Objective - French": "Le fonds cherche \u00e0 r\u00e9aliser une croissance du capital \u00e0 long terme. Le Sous-Fonds investit au moins 50% de ses actifs dans des Investissements durables, tels que d\u00e9finis par le SFDR."
        },
        "response": {
            "PRIIPS KID Objective": {
                "language": "English",
                "objectives": [
                    {
                        "objective_text": "achieve long-term capital growth",
                        "source_text": "The fund seeks to achieve long-term capital growth.",
                        "objective_type": "financial"
                    },
                    {
                        "objective_text": "invest at least 50% of assets in Sustainable Investments, as defined under SFDR",
                        "source_text": "The Sub-Fund invests at least 50% of assets in Sustainable Investments, as defined under SFDR.",
                        "objective_type": "sustainable_disclosure"
                    }
                ]
            },
            "PRIIPS KID Objective - French": {
                "language": "French",
                "objectives": [
                    {
                        "objective_text": "achieve long-term capital growth",
                        "source_text": "Le fonds cherche \u00e0 r\u00e9aliser une croissance du capital \u00e0 long terme.",
                        "objective_type": "financial"
                    },
                    {
                        "objective_text": "invest at least 50% of assets in Sustainable Investments, as defined under SFDR",
                        "source_text": "Le Sous-Fonds investit au moins 50% de ses actifs dans des Investissements durables, tels que d\u00e9finis par le SFDR.",
                        "objective_type": "sustainable_disclosure"
                    }
                ]
            }
        }
    },
    {
        "fund_name": "Example Market Exposure HOW Stripped",
        "columns": {
            "PRIIPS KID Objective": "The fund aims to achieve long-term performance through exposure to European equity markets, investing in leading growth companies in their sectors."
        },
        "response": {
            "PRIIPS KID Objective": {
                "language": "English",
                "objectives": [
                    {
                        "objective_text": "achieve long-term performance",
                        "source_text": "The fund aims to achieve long-term performance through exposure to European equity markets, investing in leading growth companies in their sectors.",
                        "objective_type": "financial"
                    }
                ]
            }
        }
    },
    {
        "fund_name": "Example Investment Policy Only",
        "columns": {
            "PRIIPS KID Objective": "The fund invests primarily in European high-dividend yield companies' equities through an active stock selection process."
        },
        "response": {
            "PRIIPS KID Objective": {
                "language": "English",
                "objectives": []
            }
        }
    },
    {
        "fund_name": "Example Labeled Constraint as Objective",
        "columns": {
            "PRIIPS KID Objective": "The fund seeks long-term capital growth. The fund also aims to maintain an ESG score above that of the MSCI World Index. The fund's portfolio is broadly diversified across global equities."
        },
        "response": {
            "PRIIPS KID Objective": {
                "language": "English",
                "objectives": [
                    {
                        "objective_text": "achieve long-term capital growth",
                        "source_text": "The fund seeks long-term capital growth.",
                        "objective_type": "financial"
                    },
                    {
                        "objective_text": "maintain an ESG score above that of the MSCI World Index",
                        "source_text": "The fund also aims to maintain an ESG score above that of the MSCI World Index.",
                        "objective_type": "sustainable"
                    }
                ]
            }
        }
    }
]

In [29]:
PASS2_SYSTEM_PROMPT = """You are consolidating fund objective extractions that were made independently from multiple regulatory text columns for the same European mutual fund.

You will receive a JSON object where each key is a column name, and the value contains:
- "language": the language of that column
- "objectives": a list of objectives extracted from that column, each with:
  - "objective_text": concise English summary of the goal
  - "source_text": verbatim excerpt from the source in original language (audit trail)
  - "objective_type": "financial", "sustainable", or "sustainable_disclosure"
    (sustainable_disclosure = a sustainability clause that uses regulatory language
    and may be a mandatory disclosure rather than a fund-chosen objective — carry
    this classification through unchanged; do not promote it to "sustainable")

YOUR TASK:
1. MATCH equivalent objectives across columns/languages. The same objective may appear in English, French, German, Swedish, etc. Compare objective_text values — since these are clean English summaries, matching across languages is straightforward.
2. DEDUPLICATE: If multiple columns express the same objective, keep it only once.
3. For each unique objective, select the BEST objective_text phrasing — prefer a native English source if available; otherwise use or improve the translation. Keep the source_text from the most authoritative source column.
4. Classify each as "financial", "sustainable", or "sustainable_disclosure".
5. Record which columns contained this objective (for traceability).

MATCHING GUIDANCE:
- "long-term capital growth" and "achieve long-term capital growth" = SAME objective
- "outperform the benchmark" and "exceed the benchmark index" = SAME objective (minor wording variation)
- "achieve capital growth" and "achieve capital growth" plus "outperform the benchmark" — the second set contains TWO objectives; match the first and keep the second as separate
- Be generous in matching across languages but strict about not merging genuinely different objectives
- Do NOT recombine objectives that were correctly split in Pass 1. If one column produced a single combined entry while another correctly produced two separate entries, the split form takes precedence.
- Intensity qualifiers do NOT make objectives distinct: "generate an increase in value" and "generate a high increase in value" express the same goal — merge them, keeping the more specific phrasing. The same applies to "maximum", "attractive", "strong", "above-average" — they are degree modifiers, not separate objectives.
- Core vocabulary equivalences — treat these as the same financial objective and deduplicate: "capital growth" = "capital appreciation" = "increase in value" = "value growth" = "asset growth". If two objectives differ only in which of these terms they use and have the same time horizon, they are duplicates.

OUTPUT FORMAT:
{
  "consolidated_objectives": [
    {
      "objective_number": 1,
      "objective_text": "the final concise English statement of this objective",
      "source_text": "verbatim excerpt from the most authoritative source column",
      "objective_type": "financial" or "sustainable" or "sustainable_disclosure",
      "found_in_columns": ["PRIIPS KID Objective", "PRIIPS KID Objective - French", ...],
      "match_notes": "brief note on how columns were matched, or null if only found in one column"
    }
  ],
  "consolidation_notes": "any important notes about the consolidation process"
}

If Pass 1 found NO objectives in ANY column:
{
  "consolidated_objectives": [],
  "consolidation_notes": "NOT IDENTIFIED — no objectives found in any column"
}
"""

PASS2_FEW_SHOT = [
    {
        "fund_name": "Example Multilingual Fund",
        "pass1_data": {
            "PRIIPS KID Objective": {
                "language": "English",
                "objectives": [
                    {"objective_text": "achieve capital growth", "source_text": "The fund seeks to achieve capital growth and to outperform the benchmark.", "objective_type": "financial"},
                    {"objective_text": "outperform the benchmark", "source_text": "The fund seeks to achieve capital growth and to outperform the benchmark.", "objective_type": "financial"}
                ]
            },
            "PRIIPS KID Objective - French": {
                "language": "French",
                "objectives": [
                    {"objective_text": "achieve capital growth", "source_text": "Le fonds cherche \u00e0 r\u00e9aliser une croissance du capital et \u00e0 surperformer l'indice de r\u00e9f\u00e9rence.", "objective_type": "financial"},
                    {"objective_text": "outperform the benchmark index", "source_text": "Le fonds cherche \u00e0 r\u00e9aliser une croissance du capital et \u00e0 surperformer l'indice de r\u00e9f\u00e9rence.", "objective_type": "financial"}
                ]
            },
            "PRIIPS KID Objective - German": {
                "language": "German",
                "objectives": [
                    {"objective_text": "achieve capital growth", "source_text": "Der Fonds strebt Kapitalwachstum an und versucht, die Benchmark zu \u00fcbertreffen.", "objective_type": "financial"},
                    {"objective_text": "outperform the benchmark", "source_text": "Der Fonds strebt Kapitalwachstum an und versucht, die Benchmark zu \u00fcbertreffen.", "objective_type": "financial"},
                    {"objective_text": "reduce greenhouse gas emissions", "source_text": "Nachhaltiges Anlageziel des Fonds ist die Reduzierung der Treibhausgasemissionen.", "objective_type": "sustainable"}
                ]
            }
        },
        "response": {
            "consolidated_objectives": [
                {
                    "objective_number": 1,
                    "objective_text": "achieve capital growth",
                    "source_text": "The fund seeks to achieve capital growth and to outperform the benchmark.",
                    "objective_type": "financial",
                    "found_in_columns": ["PRIIPS KID Objective", "PRIIPS KID Objective - French", "PRIIPS KID Objective - German"],
                    "match_notes": "Same objective across all three language columns"
                },
                {
                    "objective_number": 2,
                    "objective_text": "outperform the benchmark",
                    "source_text": "The fund seeks to achieve capital growth and to outperform the benchmark.",
                    "objective_type": "financial",
                    "found_in_columns": ["PRIIPS KID Objective", "PRIIPS KID Objective - French", "PRIIPS KID Objective - German"],
                    "match_notes": "Same benchmark-beating objective across all three languages"
                },
                {
                    "objective_number": 3,
                    "objective_text": "reduce greenhouse gas emissions",
                    "source_text": "Nachhaltiges Anlageziel des Fonds ist die Reduzierung der Treibhausgasemissionen.",
                    "objective_type": "sustainable",
                    "found_in_columns": ["PRIIPS KID Objective - German"],
                    "match_notes": "Sustainability objective found only in German column"
                }
            ],
            "consolidation_notes": "Two financial objectives matched across all three languages. One sustainability objective found only in the German column."
        }
    },
    {
        "fund_name": "Example Sustainable Disclosure Fund",
        "pass1_data": {
            "PRIIPS KID Objective": {
                "language": "English",
                "objectives": [
                    {
                        "objective_text": "achieve long-term capital growth",
                        "source_text": "The fund seeks to achieve long-term capital growth.",
                        "objective_type": "financial"
                    },
                    {
                        "objective_text": "invest at least 50% of assets in Sustainable Investments, as defined under SFDR",
                        "source_text": "The Sub-Fund invests at least 50% of assets in Sustainable Investments, as defined under SFDR.",
                        "objective_type": "sustainable_disclosure"
                    }
                ]
            },
            "PRIIPS KID Objective - French": {
                "language": "French",
                "objectives": [
                    {
                        "objective_text": "achieve long-term capital growth",
                        "source_text": "Le fonds cherche \u00e0 r\u00e9aliser une croissance du capital \u00e0 long terme.",
                        "objective_type": "financial"
                    },
                    {
                        "objective_text": "invest at least 50% of assets in Sustainable Investments, as defined under SFDR",
                        "source_text": "Le Sous-Fonds investit au moins 50% de ses actifs dans des Investissements durables, tels que d\u00e9finis par le SFDR.",
                        "objective_type": "sustainable_disclosure"
                    }
                ]
            }
        },
        "response": {
            "consolidated_objectives": [
                {
                    "objective_number": 1,
                    "objective_text": "achieve long-term capital growth",
                    "source_text": "The fund seeks to achieve long-term capital growth.",
                    "objective_type": "financial",
                    "found_in_columns": ["PRIIPS KID Objective", "PRIIPS KID Objective - French"],
                    "match_notes": "Same financial objective in English and French columns"
                },
                {
                    "objective_number": 2,
                    "objective_text": "invest at least 50% of assets in Sustainable Investments, as defined under SFDR",
                    "source_text": "The Sub-Fund invests at least 50% of assets in Sustainable Investments, as defined under SFDR.",
                    "objective_type": "sustainable_disclosure",
                    "found_in_columns": ["PRIIPS KID Objective", "PRIIPS KID Objective - French"],
                    "match_notes": "Same regulatory disclosure in both languages; classification carried through as sustainable_disclosure unchanged"
                }
            ],
            "consolidation_notes": "One financial objective and one sustainable_disclosure item. The sustainable_disclosure classification is preserved from Pass 1 for human review."
        }
    }
]


## Cells 5–8 — Functions (no edits needed)

In [31]:
def robust_json_parse(text):
    if not isinstance(text, str):
        return None
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r'^```\w*\n?', '', cleaned)
        cleaned = re.sub(r'\n?```\s*$', '', cleaned)
        cleaned = cleaned.strip()
    for old, new in [('\u201e','\u201E'), ('\u201c','\u201C'), ('\u201d','\u201D'),
                     ('\u00ab','\u00AB'), ('\u00bb','\u00BB'),
                     ('\u201a','\u201A'), ('\u2018','\u2018'), ('\u2019','\u2019')]:
        cleaned = cleaned.replace(old, f'\\u{ord(new):04X}')
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    def fix_strings(m):
        s = m.group(0)
        for ch, esc in [('\n','\\n'),('\r','\\r'),('\t','\\t')]:
            s = s.replace(ch, esc)
        return s
    fixed = re.sub(r'"(?:[^"\\]|\\.)*"', fix_strings, cleaned, flags=re.DOTALL)
    try:
        return json.loads(fixed)
    except json.JSONDecodeError:
        pass
    m = re.search(r'\{.*\}', fixed, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0))
        except json.JSONDecodeError:
            pass
    return None


def get_nonempty_columns(row, objective_columns):
    cols = {}
    for col in objective_columns:
        if col in row.index:
            val = row[col]
            if pd.notna(val) and str(val).strip() not in ['-', 'Not available', '']:
                cols[col] = str(val)
    return cols


def get_source_columns_text(fund_id, df_source, objective_columns):
    fund_row = df_source[df_source['FundId'] == fund_id]
    if fund_row.empty:
        return {}
    return get_nonempty_columns(fund_row.iloc[0], objective_columns)


print("Utilities ready.")


Utilities ready.


In [32]:
def pass1_extract(fund_name, fund_id, columns_dict):
    if not columns_dict:
        return {"_error": "No non-empty columns available"}

    columns_text = "\n\n".join(
        [f"=== Column: {col} ===\n{val}" for col, val in columns_dict.items()]
    )
    user_prompt = f"Fund ID: {fund_id}\nFund Name: {fund_name}\n\n{columns_text}"

    messages = []
    for ex in PASS1_FEW_SHOT:
        ex_text = "\n\n".join(
            [f"=== Column: {col} ===\n{val}" for col, val in ex["columns"].items()]
        )
        messages.append({"role": "user",
                          "content": f"Fund Name: {ex['fund_name']}\n\n{ex_text}"})
        messages.append({"role": "assistant",
                          "content": json.dumps(ex["response"], indent=2)})
    messages.append({"role": "user", "content": user_prompt})

    try:
        client = anthropic.Anthropic()
        response = client.messages.create(
            model=MODEL, max_tokens=8000, temperature=0,
            system=PASS1_SYSTEM_PROMPT, messages=messages
        )
        print(f"   [{fund_name}] in: {response.usage.input_tokens}  out: {response.usage.output_tokens}")
        parsed = robust_json_parse(response.content[0].text)
        return parsed if parsed is not None else {"_error": f"JSON parse failed: {response.content[0].text[:200]}"}
    except Exception as e:
        print(f"   Error [{fund_name}]: {e}")
        return {"_error": str(e)}


print("Pass 1 functions ready.")


Pass 1 functions ready.


In [33]:
def pass2_consolidate(fund_name, fund_id, pass1_data):
    cols_with_data = {
        col: data for col, data in pass1_data.items()
        if not col.startswith("_")
        and isinstance(data, dict)
        and len(data.get("objectives", [])) > 0
    }
    if not cols_with_data:
        return {"consolidated_objectives": [],
                "consolidation_notes": "NOT IDENTIFIED — Pass 1 found no objectives in any column"}

    user_prompt = (
        f"Fund ID: {fund_id}\nFund Name: {fund_name}\n\n"
        f"Pass 1 extractions (per-column):\n{json.dumps(cols_with_data, indent=2)}"
    )
    messages = []
    for ex in PASS2_FEW_SHOT:
        messages.append({"role": "user",
                          "content": f"Fund Name: {ex['fund_name']}\n\nPass 1 extractions (per-column):\n{json.dumps(ex['pass1_data'], indent=2)}"})
        messages.append({"role": "assistant",
                          "content": json.dumps(ex["response"], indent=2)})
    messages.append({"role": "user", "content": user_prompt})

    try:
        client = anthropic.Anthropic()
        response = client.messages.create(
            model=MODEL, max_tokens=4000, temperature=0,
            system=PASS2_SYSTEM_PROMPT, messages=messages
        )
        print(f"   [{fund_name}] in: {response.usage.input_tokens}  out: {response.usage.output_tokens}")
        parsed = robust_json_parse(response.content[0].text)
        return parsed if parsed is not None else {"_error": f"JSON parse failed: {response.content[0].text[:200]}"}
    except Exception as e:
        print(f"   Error [{fund_name}]: {e}")
        return {"_error": str(e)}


print("Pass 2 functions ready.")


Pass 2 functions ready.


In [34]:
def pass3_verify(fund_name, fund_id, pass2_objectives, source_columns):
    if not pass2_objectives:
        return {"verified_objectives": [], "overall_confidence": "none",
                "verification_summary": "No objectives to verify"}

    source_text = "\n\n".join(
        [f"=== Column: {col} ===\n{val}" for col, val in source_columns.items()]
    )
    user_prompt = (
        f"Fund ID: {fund_id}\nFund Name: {fund_name}\n\n"
        f"CONSOLIDATED OBJECTIVES FROM PASS 2:\n{json.dumps(pass2_objectives, indent=2)}\n\n"
        f"ORIGINAL SOURCE TEXT (all available columns):\n{source_text}"
    )
    messages = [{"role": "user", "content": user_prompt}]

    try:
        client = anthropic.Anthropic()
        response = client.messages.create(
            model=MODEL, max_tokens=8000, temperature=0,
            system=PASS3_SYSTEM_PROMPT, messages=messages
        )
        print(f"   [{fund_name}] in: {response.usage.input_tokens}  out: {response.usage.output_tokens}")
        parsed = robust_json_parse(response.content[0].text)
        return parsed if parsed is not None else {"_error": f"JSON parse failed: {response.content[0].text[:200]}"}
    except Exception as e:
        print(f"   Error [{fund_name}]: {e}")
        return {"_error": str(e)}


print("Pass 3 functions ready.")


Pass 3 functions ready.


## Cell 9 — Load data

In [35]:
print("Loading source data...")
df = pd.read_excel(INPUT_FILE)
print(f"  {len(df)} funds, {len(df.columns)} columns")

if TARGET_IDS:
    df_sample = df[df['FundId'].isin(TARGET_IDS)].copy()
    print(f"  Filtered to {len(df_sample)} target funds")
elif SAMPLE_SIZE:
    df_sample = df.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE)
    print(f"  Sampled {len(df_sample)} funds (random_state={RANDOM_STATE})")
else:
    df_sample = df.copy()
    print(f"  Running on all {len(df_sample)} funds")


Loading source data...
  5680 funds, 133 columns
  Sampled 50 funds (random_state=42)


## Cells 10–12 — Run pipeline

Each cell prints live progress. Re-run individual cells to retry a pass.

In [36]:
# ── PASS 1: Extract objectives from every column ───────────────────────────
print("=" * 70)
print("PASS 1 — EXTRACT")
print("=" * 70)

pass1_results = []
for idx in tqdm(range(len(df_sample)), desc="Pass 1"):
    row      = df_sample.iloc[idx]
    fund_id  = row['FundId']
    fund_name = row['Name']
    cols_dict = get_nonempty_columns(row, OBJECTIVE_COLUMNS)
    result   = pass1_extract(fund_name, fund_id, cols_dict)
    pass1_results.append({
        'FundId': fund_id, 'Fund_Name': fund_name,
        'columns_sent': list(cols_dict.keys()),
        'num_columns_sent': len(cols_dict),
        'pass1_raw': result,
    })
    if idx > 0 and idx % 50 == 0:
        time.sleep(0.5)

pass1_df = pd.DataFrame(pass1_results)

# Quick summary
errors  = sum(1 for r in pass1_results if '_error' in r['pass1_raw'])
with_obj = sum(1 for r in pass1_results
               if any(len(v.get('objectives',[])) > 0
                      for k, v in r['pass1_raw'].items()
                      if isinstance(v, dict) and not k.startswith('_')))
print(f"\nPass 1 done: {len(pass1_df)} funds | errors: {errors} | with ≥1 obj: {with_obj}")


PASS 1 — EXTRACT


Pass 1:   2%|▏         | 1/50 [00:01<01:25,  1.75s/it]

   [eQ Europe Dividend 1 K] in: 7184  out: 64


Pass 1:   4%|▍         | 2/50 [00:12<05:48,  7.27s/it]

   [Deep Research Equity Fd SICAV A] in: 8626  out: 864


...

   [Sidera Funds Christian Equity A EUR Acc] in: 8094  out: 819


Pass 1:  98%|█████████▊| 49/50 [12:44<00:16, 16.23s/it]

   [Robeco 3D US Equity ETF USD Acc] in: 14752  out: 1523


Pass 1: 100%|██████████| 50/50 [12:47<00:00, 15.36s/it]

   [GS Emerging Europe Equity Fund (NL) P] in: 8119  out: 254

Pass 1 done: 50 funds | errors: 1 | with ≥1 obj: 48


In [37]:
# ── PASS 2: Consolidate & deduplicate across columns ───────────────────────
print("=" * 70)
print("PASS 2 — CONSOLIDATE")
print("=" * 70)

pass2_results = []
for _, row in tqdm(pass1_df.iterrows(), total=len(pass1_df), desc="Pass 2"):
    fund_id   = row['FundId']
    fund_name = row['Fund_Name']
    p1_data   = row['pass1_raw']

    if '_error' in p1_data:
        pass2_results.append({'FundId': fund_id, 'Fund_Name': fund_name,
            'pass2_raw': {'_error': f"Skipped — Pass 1 error: {p1_data['_error']}"}})
        continue

    result = pass2_consolidate(fund_name, fund_id, p1_data)
    pass2_results.append({'FundId': fund_id, 'Fund_Name': fund_name, 'pass2_raw': result})

pass2_df = pd.DataFrame(pass2_results)

total_obj = sum(len(r['pass2_raw'].get('consolidated_objectives', []))
                for r in pass2_results if '_error' not in r['pass2_raw'])
print(f"\nPass 2 done: {len(pass2_df)} funds | total objectives extracted: {total_obj}")


PASS 2 — CONSOLIDATE


Pass 2:   4%|▍         | 2/50 [00:07<03:09,  3.95s/it]

   [Deep Research Equity Fd SICAV A] in: 3496  out: 477


Pass 2:   6%|▌         | 3/50 [00:13<03:37,  4.63s/it]

   [K Investor Friendly I EUR] in: 2989  out: 418


...

   [Sidera Funds Christian Equity A EUR Acc] in: 3459  out: 483


Pass 2:  98%|█████████▊| 49/50 [06:36<00:07,  7.73s/it]

   [Robeco 3D US Equity ETF USD Acc] in: 4338  out: 440


Pass 2: 100%|██████████| 50/50 [06:39<00:00,  7.99s/it]

   [GS Emerging Europe Equity Fund (NL) P] in: 2896  out: 197

Pass 2 done: 50 funds | total objectives extracted: 83


In [38]:
# ── PASS 3: Verify & save ───────────────────────────────────────────────────
print("=" * 70)
print("PASS 3 — VERIFY")
print("=" * 70)

# Load source data for verification
df_source = pd.read_excel(INPUT_FILE)

pass3_results = []
for _, row in tqdm(pass2_df.iterrows(), total=len(pass2_df), desc="Pass 3"):
    fund_id   = row['FundId']
    fund_name = row['Fund_Name']
    p2_data   = row['pass2_raw']

    if '_error' in p2_data:
        pass3_results.append({'FundId': fund_id, 'Fund_Name': fund_name,
            'pass3_raw': {'_error': f"Skipped — Pass 2 error: {p2_data['_error']}"}})
        continue

    objectives = p2_data.get('consolidated_objectives', [])
    if not objectives:
        pass3_results.append({'FundId': fund_id, 'Fund_Name': fund_name,
            'pass3_raw': {'verified_objectives': [], 'overall_confidence': 'none',
                           'verification_summary': 'No objectives from Pass 2'}})
        continue

    source_cols = get_source_columns_text(fund_id, df_source, OBJECTIVE_COLUMNS)
    result = pass3_verify(fund_name, fund_id, objectives, source_cols)
    pass3_results.append({'FundId': fund_id, 'Fund_Name': fund_name, 'pass3_raw': result})
    if len(pass3_results) % 50 == 0:
        time.sleep(0.5)

pass3_df = pd.DataFrame(pass3_results)

# ── Flatten into final output ───────────────────────────────────────────────
final_rows = []
for _, row in pass3_df.iterrows():
    raw  = row['pass3_raw']
    base = {'FundId': row['FundId'], 'Fund_Name': row['Fund_Name']}

    if '_error' in raw:
        base.update({'Number_of_Objectives': 0, 'Overall_Confidence': 'error',
                     'Verification_Summary': raw['_error'], 'Has_Flagged': False})
        final_rows.append(base); continue

    objs = raw.get('verified_objectives', [])
    base['Number_of_Objectives'] = len(objs)
    base['Overall_Confidence']   = raw.get('overall_confidence', '')
    base['Verification_Summary'] = raw.get('verification_summary', '')
    base['Has_Flagged']          = any(o.get('verification_status') == 'FLAGGED' for o in objs)

    for i in range(5):
        if i < len(objs):
            o = objs[i]
            base[f'Objective_{i+1}']             = o.get('objective_text', '')
            base[f'Objective_{i+1}_Source']       = o.get('source_text', '')
            base[f'Objective_{i+1}_Type']         = o.get('objective_type', '')
            base[f'Objective_{i+1}_Status']       = o.get('verification_status', '')
            base[f'Objective_{i+1}_Verified_In']  = o.get('verified_in_column', '')
            base[f'Objective_{i+1}_Type_Changed'] = o.get('type_changed', False)
            base[f'Objective_{i+1}_Notes']        = o.get('verification_notes', '')
        else:
            for suffix in ['_Source','_Type','_Status','_Verified_In','_Type_Changed','_Notes','']:
                base[f'Objective_{i+1}{suffix}'] = None
    final_rows.append(base)

final_df = pd.DataFrame(final_rows)

# ── Save outputs ────────────────────────────────────────────────────────────
ts = pd.Timestamp.now().strftime("%Y%m%d_%H%M")
n  = len(final_df)

# Intermediate: Pass 1 raw (useful for debugging)
p1_save = pass1_df.copy()
p1_save['pass1_raw']    = p1_save['pass1_raw'].apply(json.dumps)
p1_save['columns_sent'] = p1_save['columns_sent'].apply(json.dumps)
p1_save.to_excel(OUTPUT_DIR / f'Pass1_Extract_{n}_funds_{ts}.xlsx', index=False)

# Intermediate: Pass 2 flattened
p2_flat_rows = []
for _, row in pass2_df.iterrows():
    raw  = row['pass2_raw']
    base = {'FundId': row['FundId'], 'Fund_Name': row['Fund_Name']}
    if '_error' in raw:
        base.update({'Number_of_Objectives': 0, 'Consolidation_Notes': raw['_error']})
    else:
        objs = raw.get('consolidated_objectives', [])
        base['Number_of_Objectives'] = len(objs)
        base['Consolidation_Notes']  = raw.get('consolidation_notes', '')
        for i, o in enumerate(objs[:5]):
            base[f'Objective_{i+1}']         = o.get('objective_text', '')
            base[f'Objective_{i+1}_Source']   = o.get('source_text', '')
            base[f'Objective_{i+1}_Type']     = o.get('objective_type', '')
            base[f'Objective_{i+1}_Columns']  = ', '.join(o.get('found_in_columns', []))
    p2_flat_rows.append(base)
pd.DataFrame(p2_flat_rows).to_excel(OUTPUT_DIR / f'Pass2_Consolidated_{n}_funds_{ts}.xlsx', index=False)

# Final verified output
final_path = OUTPUT_DIR / f'FINAL_Verified_{n}_funds_{ts}.xlsx'
final_df.to_excel(final_path, index=False, engine='openpyxl')

# ── Summary ─────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("PIPELINE COMPLETE")
print("=" * 70)
total     = len(final_df)
with_obj  = (final_df['Number_of_Objectives'] > 0).sum()
flagged   = final_df['Has_Flagged'].sum()
all_types = [final_df[f'Objective_{i}_Type'].dropna().tolist()
             for i in range(1, 6)]
all_types = [t for sub in all_types for t in sub]
counts    = Counter(all_types)

print(f"  Funds processed:       {total}")
print(f"  Funds with objectives: {with_obj} ({with_obj/total*100:.0f}%)")
print(f"  Funds with FLAGGED:    {flagged}")
print(f"  Financial objectives:  {counts.get('financial', 0)}")
print(f"  Sustainable:           {counts.get('sustainable', 0)}")
print(f"  Sustainable disclosure:{counts.get('sustainable_disclosure', 0)}")
print(f"\n  Saved: {final_path.name}")


PASS 3 — VERIFY


Pass 3:   4%|▍         | 2/50 [00:09<03:56,  4.93s/it]

   [Deep Research Equity Fd SICAV A] in: 2915  out: 531


Pass 3:   6%|▌         | 3/50 [00:17<04:50,  6.18s/it]

   [K Investor Friendly I EUR] in: 1177  out: 570


...

   [Sidera Funds Christian Equity A EUR Acc] in: 2398  out: 611


Pass 3:  98%|█████████▊| 49/50 [07:58<00:10, 10.61s/it]

   [Robeco 3D US Equity ETF USD Acc] in: 8979  out: 439
   [GS Emerging Europe Equity Fund (NL) P] in: 2177  out: 393


Pass 3: 100%|██████████| 50/50 [08:06<00:00,  9.73s/it]


PIPELINE COMPLETE
  Funds processed:       50
  Funds with objectives: 48 (96%)
  Funds with FLAGGED:    0
  Financial objectives:  70
  Sustainable:           9
  Sustainable disclosure:4

  Saved: FINAL_Verified_50_funds_20260610_1319.xlsx
